In [ ]:
# Cell 1 - Install/imports + env load
# Fresh venv? Uncomment once:
# %pip install -r requirements.txt

import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = Path.cwd()
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from exa_demo.artifacts import ExperimentArtifactWriter
from exa_demo.cache import SqliteCacheStore
from exa_demo.client import build_exa_payload as _build_exa_payload
from exa_demo.client import exa_search_people as _exa_search_people
from exa_demo.config import default_config, default_pricing, load_runtime_state
from exa_demo.cost_model import estimate_cost_from_pricing as _estimate_cost_from_pricing_base
from exa_demo.evaluation import DEFAULT_RELEVANCE_KEYWORDS, evaluate_batch_queries, load_benchmark_queries
from exa_demo.reporting import build_cost_projections, build_qualitative_notes, recommendation
from exa_demo.safety import extract_preview as _module_extract_preview
from exa_demo.safety import redact_text as _module_redact_text

load_dotenv()  # loads .env from notebook working directory

runtime = load_runtime_state()
EXA_API_KEY = runtime.exa_api_key
EXA_SMOKE_NO_NETWORK = runtime.smoke_no_network
RUN_ID = runtime.run_id

if EXA_SMOKE_NO_NETWORK:
    print("Mode: smoke (no network)")
else:
    print("Mode: live Exa API")

print(f"RUN_ID: {RUN_ID}")


In [ ]:
# Cell 2 - Config

CONFIG = default_config()
PRICING = default_pricing()

print(json.dumps(CONFIG, indent=2))


In [ ]:
# Cell 3 - Exa call wrapper

def _estimate_cost_from_pricing(payload: Dict[str, Any], num_results: int) -> float:
    return _estimate_cost_from_pricing_base(
        payload,
        num_results,
        PRICING,
        int(CONFIG["max_supported_results_for_estimate"]),
    )


def redact_text(s: Optional[str]) -> Optional[str]:
    return _module_redact_text(s, enabled=CONFIG.get("redact_emails_phones", True))


def extract_preview(result: Dict[str, Any], max_chars: int = 280) -> str:
    return _module_extract_preview(
        result,
        max_chars=max_chars,
        redact_enabled=CONFIG.get("redact_emails_phones", True),
    )


def build_exa_payload(query: str, *, num_results: Optional[int] = None) -> Dict[str, Any]:
    return _build_exa_payload(query, CONFIG, num_results=num_results)


def exa_search_people(query: str, *, num_results: Optional[int] = None):
    return _exa_search_people(
        query,
        config=CONFIG,
        pricing=PRICING,
        exa_api_key=EXA_API_KEY,
        smoke_no_network=EXA_SMOKE_NO_NETWORK,
        run_id=RUN_ID,
        cache_store=_cache_store(),
        num_results=num_results,
    )


In [ ]:
# Cell 4 - Cache wrapper (sqlite) + run-scoped budget enforcement


def _cache_store() -> SqliteCacheStore:
    return SqliteCacheStore(CONFIG["sqlite_path"], CONFIG["cache_ttl_hours"])


def ledger_summary(run_id: Optional[str] = None) -> pd.DataFrame:
    return _cache_store().ledger_summary(run_id=run_id)


def spend_so_far(run_id: Optional[str] = None) -> Dict[str, float]:
    return _cache_store().spend_so_far(run_id=run_id)


ARTIFACT_WRITER = ExperimentArtifactWriter(
    run_id=RUN_ID,
    config=CONFIG,
    pricing=PRICING,
)

print("Cache ready:", CONFIG["sqlite_path"])
print("Experiment artifacts:", ARTIFACT_WRITER.artifact_dir)
print("Run metrics:", spend_so_far(run_id=RUN_ID))
print("All-time metrics:", spend_so_far())


In [ ]:
# Cell 5 - Single query demo

demo_query = "Florida property insurance attorney hurricane Ian appraisal dispute site:linkedin.com"
resp_demo, meta_demo = exa_search_people(demo_query, num_results=CONFIG["num_results"])

print("Single-query demo")
print("  run_id:", RUN_ID)
print("  cache_hit:", meta_demo.cache_hit)
print("  estimated_cost_usd:", meta_demo.estimated_cost_usd)
print("  actual_cost_usd:", meta_demo.actual_cost_usd)
print("  request_id:", meta_demo.request_id)

results_demo = resp_demo.get("results", []) if isinstance(resp_demo, dict) else []
rows_demo = []
for result in results_demo[: int(CONFIG["num_results"])]:
    rows_demo.append(
        {
            "title": redact_text(result.get("title")),
            "url": result.get("url"),
            "preview": extract_preview(result, max_chars=280),
            "highlightScores": result.get("highlightScores"),
        }
    )

df_demo = pd.DataFrame(rows_demo)
print("Rows returned:", len(df_demo))
df_demo


In [ ]:
# Cell 6 - Batch query test suite (insurance / CAT)

# Swap this list only; the rest of the notebook will continue to work.
TEST_QUERIES = load_benchmark_queries()
RELEVANCE_KEYWORDS = list(DEFAULT_RELEVANCE_KEYWORDS)

df_batch = evaluate_batch_queries(
    TEST_QUERIES,
    search_people=exa_search_people,
    num_results=int(CONFIG["num_results"]),
    relevance_keywords=RELEVANCE_KEYWORDS,
    redact_text=redact_text,
    extract_preview=extract_preview,
    record_query=ARTIFACT_WRITER.record_query,
)

print(f"Batch queries executed: {len(df_batch)}")
df_batch


In [ ]:
# Cell 7 - Summary table + qualitative notes

summary = spend_so_far(run_id=RUN_ID)
summary_all = spend_so_far()

df_ledger_run = ledger_summary(run_id=RUN_ID)

df_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "request_count": summary["request_count"],
    "cache_hits": summary["cache_hits"],
    "uncached_calls": summary["uncached_calls"],
    "spent_usd": summary["spent_usd"],
    "avg_cost_per_uncached_query": summary["avg_cost_per_uncached_query"],
}])

print(f"Request count: {summary['request_count']}")
print(f"Cache hits vs uncached calls: {summary['cache_hits']} vs {summary['uncached_calls']}")
print(f"Spent so far (run {RUN_ID}): ${summary['spent_usd']:.4f}")
print(f"Avg cost per uncached query: ${summary['avg_cost_per_uncached_query']:.4f}")
print(f"All-time spend across runs: ${summary_all['spent_usd']:.4f}")

print("\nSummary table")
try:
    print(df_summary.to_markdown(index=False))
except Exception:
    print(df_summary)

qualitative_notes = build_qualitative_notes(
    df_batch,
    CONFIG,
    smoke_no_network=EXA_SMOKE_NO_NETWORK,
)

print("\nQualitative notes")
for note in qualitative_notes:
    print("-", note)

review_cols = [
    "query",
    "top_title",
    "top_url",
    "top_preview",
    "linkedin_present",
    "relevance_keywords_present",
    "cache_hit",
    "actual_cost_usd",
    "est_cost_usd",
]

df_batch[review_cols]


In [ ]:
# Cell 8 - Cost estimate + projections

observed_spent = float(summary.get("spent_usd", 0.0))
projections = build_cost_projections(summary, config=CONFIG, pricing=PRICING)

print(f"Spent so far (run {RUN_ID}): ${observed_spent:.4f}")
print(f"Projection basis: {projections['projection_basis']}")
print(f"Projected cost for 100 queries:   ${projections['projected_100_queries_usd']:.4f}")
print(f"Projected cost for 1,000 queries: ${projections['projected_1000_queries_usd']:.4f}")
print(f"Projected cost for 10,000 queries:${projections['projected_10000_queries_usd']:.4f}")

pd.DataFrame([projections])


In [ ]:
# Cell 9 - Decision rubric + recommendation integration points

RUBRIC = {
    "Search quality (people relevance)": "Do results return the right categories of professionals for CAT-loss / claim-dispute work?",
    "Coverage": "Does it find credible candidates beyond obvious first-page names?",
    "Evidence quality": "Are highlights/snippets sufficient for triage without defaulting to full text?",
    "Latency": "Is response time acceptable for analyst workflows?",
    "Cost": "Is unit cost predictable and within budget at projected volumes?",
    "Safety/compliance": "Can outputs stay public/professional only (no doxxing/contact harvesting)?",
    "Repeatability": "Do cached reruns reliably reproduce behavior without rebilling?",
}

print("Decision rubric")
for criterion, prompt in RUBRIC.items():
    print(f"- [ ] {criterion}: {prompt}")


rec = recommendation(
    summary,
    df_batch,
    run_id=RUN_ID,
    budget_cap_usd=float(CONFIG["budget_cap_usd"]),
    smoke_no_network=EXA_SMOKE_NO_NETWORK,
)
print("\nRecommendation output")
print(json.dumps(rec, indent=2))

summary_path = ARTIFACT_WRITER.write_summary(
    summary,
    projections=projections,
    recommendation_data=rec,
    batch_df=df_batch,
    qualitative_notes=qualitative_notes,
    extra={"summary_all": summary_all},
)
print("\nExperiment artifacts written to:", ARTIFACT_WRITER.artifact_dir)
print("Summary artifact:", summary_path)
